# SpectraRestore — Google Colab

**KLA PS01 · SEMICON India Hackathon 2026**  
**Team:** ChipSync · SRM Institute of Science And Technology  
Joint Blind Denoising + 2× Super-Resolution for SEM Inspection Images (`NAFNet-SR2×`)

### Workflow Overview
1. **Runtime:** Set Hardware Accelerator to **GPU** (T4 / A100 / L4) under `Runtime → Change runtime type`.
2. **Dataset:** Ensure the KLA dataset is in Google Drive under `MyDrive/SpectraRestore/data/` (or upload `train/` and `val/` pairs).
3. **Run All:** Cells will automatically clone/update the repository, set up the environment, train, evaluate, and generate validation benchmark tables & visual error heatmaps for Slide 6.

## 0 · Check GPU Acceleration

In [ ]:
!nvidia-smi
import torch
print('PyTorch version:', torch.__version__, '| CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
    print('VRAM:', f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

## 1 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2 · Get Project Code

Clones the public repository or unpacks an archive if available.

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

REPO_URL = 'https://github.com/Charan-suresh/SpectraRestore.git'
PROJECT = Path('/content/SpectraRestore')
DRIVE_ARCHIVE = Path('/content/drive/MyDrive/SpectraRestore/SpectraRestore.zip')

if (PROJECT / 'src' / 'model.py').is_file():
    print(f'Project already present at {PROJECT} — updating via git pull...')
    %cd {PROJECT}
    !git pull || true
elif DRIVE_ARCHIVE.is_file():
    print(f'Extracting supplied archive: {DRIVE_ARCHIVE}')
    with zipfile.ZipFile(DRIVE_ARCHIVE) as archive:
        archive.extractall(PROJECT)
    %cd {PROJECT}
else:
    print(f'Cloning from {REPO_URL} ...')
    !git clone {REPO_URL} {PROJECT}
    %cd {PROJECT}

assert (PROJECT / 'src' / 'model.py').is_file(), (
    f'Setup failed — src/model.py not found in {PROJECT}. Check repository access.'
)

print(f'\nCurrent Working Directory: {Path.cwd()}')

## 3 · Install Dependencies & Verify Architecture

In [ ]:
%pip install -q -r requirements.txt

import sys
from pathlib import Path
PROJECT = Path.cwd()
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from src.model import build_model
m_default = build_model('default')
m_fast = build_model('fast')
print(f'NAFNet-SR2x Default Model: {m_default.num_params()/1e6:.2f}M parameters')
print(f'NAFNet-SR2x Fast Model:    {m_fast.num_params()/1e6:.2f}M parameters')

## 4 · Dataset Setup & Verification

Expected folder structure:
```
data/
  train/
    degraded/   (128x128 / 256x256 images: png, tif, npy)
    gt/         (256x256 / 512x512 clean target images)
  val/
    degraded/
    gt/
```

In [ ]:
import os
from pathlib import Path

DRIVE_DATA = Path('/content/drive/MyDrive/SpectraRestore/data')
LOCAL_DATA = Path('data')

if LOCAL_DATA.is_symlink():
    LOCAL_DATA.unlink()

if not LOCAL_DATA.exists() and DRIVE_DATA.exists():
    print(f'Symlinking Drive dataset {DRIVE_DATA} -> {LOCAL_DATA}')
    os.symlink(DRIVE_DATA, LOCAL_DATA)

if LOCAL_DATA.exists():
    print('Dataset found at:', LOCAL_DATA.resolve())
    !find -L data -type f | sed 's|/[^/]*$||' | sort | uniq -c | sort -rn | head -10
else:
    print('Note: data/ folder not found. Please place dataset in Drive under MyDrive/SpectraRestore/data/ or in data/')

## 5 · Quick System & Pipeline Smoke Test

In [ ]:
!python scripts/smoke_test.py

## 6 · Train SpectraRestore (NAFNet-SR2×)

- **Preset:** `default` (~29M params) or `fast` (~15M params)
- **Loss:** Charbonnier (1.00) + SSIM (0.20) + FFT-L1 (0.05) + LPIPS (0.10 after warmup)
- Automatically detects existing checkpoints on Drive to resume seamlessly.

In [ ]:
from pathlib import Path

PRESET = 'default'       # default (~29M) | fast (~15M) | large (~65M)
BATCH = 4                # 4 for T4 GPU, 8 for A100/L4
ITERS = 50000            # Initial pass (or 200000 for full convergence)
GT_CROP = 256

WEIGHTS_LOCAL = Path('weights')
WEIGHTS_DRIVE = Path('/content/drive/MyDrive/SpectraRestore/weights')
WEIGHTS_LOCAL.mkdir(exist_ok=True)
WEIGHTS_DRIVE.mkdir(parents=True, exist_ok=True)

# Auto-resume from latest checkpoint on Drive if available
resume = ''
ckpts = sorted(WEIGHTS_DRIVE.glob('ckpt_*.pt'))
if ckpts:
    resume = f' --resume {ckpts[-1]}'
    print(f'Resuming from latest full checkpoint: {ckpts[-1]}')
else:
    for name in ('best.pt', 'last_ema.pt'):
        cand = WEIGHTS_DRIVE / name
        if cand.is_file():
            resume = f' --resume {cand}'
            print(f'Resuming weights from {cand}')
            break

cmd = f'''python -m src.train \\
  --data_root data \\
  --preset {PRESET} \\
  --batch_size {BATCH} \\
  --iters {ITERS} \\
  --gt_crop {GT_CROP} \\
  --num_workers 2 \\
  --val_every 1000 \\
  --save_every 2000 \\
  --log_every 50 \\
  --out_dir weights{resume}
'''
print('Executing training command:\n', cmd)
!{cmd}

## 7 · Sync Checkpoints to Google Drive

In [ ]:
import shutil
from pathlib import Path

src = Path('weights')
dst = Path('/content/drive/MyDrive/SpectraRestore/weights')
dst.mkdir(parents=True, exist_ok=True)
for f in src.glob('*.pt'):
    shutil.copy2(f, dst / f.name)
    print(f'Synced to Drive: {f.name}')

!ls -lh /content/drive/MyDrive/SpectraRestore/weights | head -10

## 8 · Run Standalone Inference (KLA evaluate.py)

Restores any input folder of degraded SEM images and preserves native filenames for evaluation matching.

In [ ]:
from pathlib import Path
import shutil

# Sync best weights from Drive if missing locally
for name in ('best.pt', 'last_ema.pt'):
    drive_w = Path('/content/drive/MyDrive/SpectraRestore/weights') / name
    local_w = Path('weights') / name
    if drive_w.is_file() and not local_w.is_file():
        Path('weights').mkdir(exist_ok=True)
        shutil.copy2(drive_w, local_w)
        print(f'Restored {name} from Drive')

INPUT = 'data/val/degraded'          # Or KLA released test folder
OUTPUT = 'outputs/val_restored'

!python evaluate.py --input_dir {INPUT} --output_dir {OUTPUT} --weights weights/best.pt

# Optional: sync restored outputs to Drive
drive_out = Path('/content/drive/MyDrive/SpectraRestore/outputs')
drive_out.mkdir(parents=True, exist_ok=True)
!cp -r {OUTPUT} {drive_out}/
print('Restored outputs synced to Drive at:', drive_out)

## 9 · Validation Benchmark & Slide 6 Results Table

Computes exact SSIM, pSNR, LPIPS, and inference latency for both the **Degraded Input Baseline** and **SpectraRestore Output** on the held-out validation split.

In [ ]:
!python scripts/benchmark_val.py --data_root data --weights weights/best.pt

## 10 · Visual Evidence & Error Heatmaps (Slide 6 Matching)

Displays 4-panel visual comparison: `[Degraded Input] → [Our Restoration] → [Ground Truth] → [|Error| Heatmap]`.

In [ ]:
import random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from evaluate import load_gray

out_dir = Path('outputs/val_restored')
deg_dir = Path('data/val/degraded')
gt_dir  = Path('data/val/gt')

restored_files = sorted([p for p in out_dir.rglob('*') if p.is_file()])
if restored_files:
    sample = random.choice(restored_files)
    stem = sample.stem
    
    deg_path = next(deg_dir.glob(stem + '.*'), None)
    gt_path = next(gt_dir.glob(stem + '.*'), None) if gt_dir.exists() else None
    
    deg_img = load_gray(deg_path) if deg_path else None
    rest_img = load_gray(sample)
    gt_img = load_gray(gt_path) if gt_path else None
    
    num_cols = 4 if gt_img is not None else 2
    fig, axs = plt.subplots(1, num_cols, figsize=(4 * num_cols, 4), dpi=150)
    
    axs[0].imshow(np.clip(deg_img, 0, 1), cmap='gray')
    axs[0].set_title('Degraded Input', fontsize=12, fontweight='bold')
    axs[0].axis('off')
    
    axs[1].imshow(np.clip(rest_img, 0, 1), cmap='gray')
    axs[1].set_title('Our Restoration', fontsize=12, fontweight='bold')
    axs[1].axis('off')
    
    if gt_img is not None:
        axs[2].imshow(np.clip(gt_img, 0, 1), cmap='gray')
        axs[2].set_title('Ground Truth', fontsize=12, fontweight='bold')
        axs[2].axis('off')
        
        # Absolute error heatmap
        err = np.abs(np.clip(rest_img, 0, 1) - np.clip(gt_img, 0, 1))
        im_err = axs[3].imshow(err, cmap='turbo', vmin=0.0, vmax=1.0)
        axs[3].set_title('|Error| Heatmap', fontsize=12, fontweight='bold')
        axs[3].axis('off')
        cbar = plt.colorbar(im_err, ax=axs[3], fraction=0.046, pad=0.04)
        cbar.ax.tick_params(labelsize=9)
    
    plt.suptitle(f'Restoration Evidence: {sample.name}', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('No restored outputs found. Run evaluation (Cell 8) first.')

## Colab Disconnect & Session Tips

1. Checkpoints and outputs are mirrored to `MyDrive/SpectraRestore/` on Drive to prevent data loss.
2. If disconnected: Reopen notebook → Mount Drive → Run All. Training will automatically pick up from the latest saved `.pt` checkpoint.
3. Design notes and technical details: see `SOLUTION.md` and `README.md`.